# Rubin anomaly detection from one Hugging Face Parquet file

This notebook reads **only one Parquet file** from the Hugging Face dataset `umbramortem/alerts_open_tools_open_sky_2026`. It uses DuckDB for efficient aggregation and avoids loading the complete repository into memory.

## Workflow

1. List the Parquet files in the Hugging Face repository.
2. Select and download one file.
3. Create a DuckDB view over that file.
4. Inspect the schema and detect likely Rubin columns.
5. Build fast object-level features.
6. Rank unusual objects with Isolation Forest.
7. Plot the highest-ranked candidates.
8. Export the ranking and original candidate rows.

> An anomaly score is a prioritization signal, not proof of a new astrophysical phenomenon. Sparse sampling, artifacts, blending, and association errors can also produce outliers.


## 0. Local environment

Recommended Conda environment:

```bash
conda create -n rubin-anomaly -c conda-forge \
  python=3.11 numpy pandas pyarrow duckdb scikit-learn \
  matplotlib huggingface_hub jupyterlab ipykernel -y
conda activate rubin-anomaly
python -m ipykernel install --user --name rubin-anomaly \
  --display-name "Python (Rubin anomaly)"
```

Select **Python (Rubin anomaly)** as the notebook kernel. For Colab only, uncomment the installation cell below.


In [ ]:
# Colab only:
# %pip install --only-binary=:all: duckdb huggingface_hub pyarrow scikit-learn

## 1. Imports and configuration


In [1]:
from pathlib import Path
from time import perf_counter
import json
import os
import sys
import warnings

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from huggingface_hub import HfApi, hf_hub_download
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_colwidth", 120)

REPO_ID = "umbramortem/alerts_open_tools_open_sky_2026"
REPO_TYPE = "dataset"
REVISION = "main"
SELECTED_FILE_INDEX = 0
MIN_OBSERVATIONS = 2
TOP_N = 12
RANDOM_STATE = 42
DUCKDB_THREADS = min(4, max(1, os.cpu_count() or 1))
DUCKDB_MEMORY_LIMIT = "4GB"

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("DuckDB:", duckdb.__version__)
print("Repository:", REPO_ID)

/Users/umbramortem/miniconda3/envs/rubin-anomaly/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.14
Executable: /Users/umbramortem/miniconda3/envs/rubin-anomaly/bin/python
DuckDB: 1.4.4
Repository: umbramortem/alerts_open_tools_open_sky_2026


## 2. List the Parquet files

Run this cell first. Change `SELECTED_FILE_INDEX` in the configuration cell if you want a different file.


In [2]:
api = HfApi(token=HF_TOKEN)
repo_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    revision=REVISION,
)

parquet_repo_files = sorted(
    filename for filename in repo_files
    if filename.lower().endswith(".parquet")
)

if not parquet_repo_files:
    raise FileNotFoundError("No Parquet files were found in the repository.")

print(f"Available Parquet files: {len(parquet_repo_files)}\n")
for index, filename in enumerate(parquet_repo_files):
    marker = "  <== selected" if index == SELECTED_FILE_INDEX else ""
    print(f"[{index}] {filename}{marker}")

if not 0 <= SELECTED_FILE_INDEX < len(parquet_repo_files):
    raise IndexError(
        f"SELECTED_FILE_INDEX must be between 0 and {len(parquet_repo_files)-1}."
    )

selected_repo_file = parquet_repo_files[SELECTED_FILE_INDEX]
print("\nSelected repository file:")
print(selected_repo_file)

Available Parquet files: 3233

[0] XRUBIN_raw_all/detections/mjd-60973p13026719-60973p63026719.parquet  <== selected
[1] XRUBIN_raw_all/detections/mjd-60973p63026719-60974p13026719.parquet
[2] XRUBIN_raw_all/detections/mjd-60974p13026719-60974p63026719.parquet
[3] XRUBIN_raw_all/detections/mjd-60974p63026719-60975p13026719.parquet
[4] XRUBIN_raw_all/detections/mjd-60975p13026719-60975p63026719.parquet
[5] XRUBIN_raw_all/detections/mjd-60975p63026719-60976p13026719.parquet
[6] XRUBIN_raw_all/detections/mjd-60976p13026719-60976p63026719.parquet
[7] XRUBIN_raw_all/detections/mjd-60976p63026719-60977p13026719.parquet
[8] XRUBIN_raw_all/detections/mjd-60977p13026719-60977p63026719.parquet
[9] XRUBIN_raw_all/detections/mjd-60977p63026719-60978p13026719.parquet
[10] XRUBIN_raw_all/detections/mjd-60978p13026719-60978p63026719.parquet
[11] XRUBIN_raw_all/detections/mjd-60978p63026719-60979p13026719.parquet
[12] XRUBIN_raw_all/detections/mjd-60979p13026719-60979p63026719.parquet
[13] XRUBIN_raw_

## 3. Download only the selected file


In [7]:
local_parquet_file = Path(
    hf_hub_download(
        repo_id=REPO_ID,
        filename=selected_repo_file,
        repo_type=REPO_TYPE,
        revision=REVISION,
        token=HF_TOKEN,
    )
)

print("Local file:", local_parquet_file)
print(f"File size: {local_parquet_file.stat().st_size / 1024**2:.2f} MB")

Local file: /Users/umbramortem/.cache/huggingface/hub/datasets--umbramortem--alerts_open_tools_open_sky_2026/snapshots/aa5281eceb31a2b9834d41a9489e6d307a9124a2/XRUBIN_raw_all/detections/mjd-60973p13026719-60973p63026719.parquet
File size: 0.03 MB


## 4. Create a DuckDB view over the selected file


In [4]:
connection = duckdb.connect(database=":memory:")
connection.execute(f"SET threads = {DUCKDB_THREADS}")
connection.execute(f"SET memory_limit = '{DUCKDB_MEMORY_LIMIT}'")
connection.execute("SET preserve_insertion_order = false")

Path("duckdb_temp").mkdir(exist_ok=True)
connection.execute("SET temp_directory = 'duckdb_temp'")


def sql_string_literal(value):
    return "'" + str(value).replace("'", "''") + "'"


local_file_sql = sql_string_literal(local_parquet_file.resolve())
connection.execute(f"""
CREATE OR REPLACE VIEW rubin_alerts AS
SELECT *
FROM read_parquet(
    {local_file_sql},
    filename = true
)
""")

print("DuckDB view created from one file.")
print("Threads:", DUCKDB_THREADS)
print("Memory limit:", DUCKDB_MEMORY_LIMIT)

DuckDB view created from one file.
Threads: 4
Memory limit: 4GB


## 5. Inspect the selected file


In [8]:
schema = connection.execute("DESCRIBE rubin_alerts").df()
display(schema)

summary = connection.execute("""
SELECT
    count(*) AS number_of_rows,
    count(DISTINCT filename) AS number_of_files
FROM rubin_alerts
""").df()
display(summary)

sample = connection.execute("SELECT * FROM rubin_alerts LIMIT 5").df()
display(sample)

,column_name,column_type,null,key,default,extra
0,oid,BIGINT,YES,None,None,None
1,measurement_id,BIGINT,YES,None,None,None
2,mjd,DOUBLE,YES,None,None,None
3,band_name,VARCHAR,YES,None,None,None
4,psfflux,FLOAT,YES,None,None,None
5,psffluxerr,FLOAT,YES,None,None,None
6,scienceflux,FLOAT,YES,None,None,None
7,sciencefluxerr,FLOAT,YES,None,None,None
8,templateflux,FLOAT,YES,None,None,None
9,templatefluxerr,FLOAT,YES,None,None,None


,number_of_rows,number_of_files
0,414,1


,oid,measurement_id,mjd,band_name,psfflux,psffluxerr,scienceflux,sciencefluxerr,templateflux,templatefluxerr,isnegative,snr,reliability,isdipole,psfflux_flag,pixelflags_bad,pixelflags_cr,pixelflags_saturated,pixelflags_streak,glint_trail,filename
0,313629129589653560,313629129589653560,60973.169745,i,4451.933105,390.692352,42350.832031,411.757233,36908.015625,6.266848,False,9.690210,0.228454,False,False,False,False,False,False,False,/Users/umbramortem/.cache/huggingface/hub/datasets--umbramortem--alerts_open_tools_open_sky_2026/blobs/a8ef6528cc796...
1,313629129556623404,313629129556623404,60973.169745,i,2495.562500,379.948944,2769.399902,396.411713,-4.087404,6.799683,False,6.470420,0.632288,False,False,False,False,False,False,False,/Users/umbramortem/.cache/huggingface/hub/datasets--umbramortem--alerts_open_tools_open_sky_2026/blobs/a8ef6528cc796...
2,313629129583362067,313629129583362067,60973.169745,i,2516.010498,470.630493,162915.484375,497.498810,159577.921875,7.125452,False,5.086137,0.275779,False,False,False,False,False,False,False,/Users/umbramortem/.cache/huggingface/hub/datasets--umbramortem--alerts_open_tools_open_sky_2026/blobs/a8ef6528cc796...
3,313629129686646795,313629129686646795,60973.170251,i,2389.827393,440.143555,24212.988281,458.937073,21841.248047,13.754683,False,5.309637,0.306016,False,False,False,False,False,False,False,/Users/umbramortem/.cache/huggingface/hub/datasets--umbramortem--alerts_open_tools_open_sky_2026/blobs/a8ef6528cc796...
4,313629129721250087,313629129721250087,60973.170251,i,2030.461914,395.169617,1894.503296,407.750946,-163.104202,12.691509,False,5.056403,0.236224,False,False,False,False,False,False,False,/Users/umbramortem/.cache/huggingface/hub/datasets--umbramortem--alerts_open_tools_open_sky_2026/blobs/a8ef6528cc796...


## 6. Detect likely Rubin columns

Inspect the output. If a mapping is incorrect, edit the manual override cell below.


In [10]:
available_columns = schema["column_name"].astype(str).tolist()


def first_existing(candidates):
    lookup = {column.lower(): column for column in available_columns}
    return next(
        (lookup[name.lower()] for name in candidates if name.lower() in lookup),
        None,
    )


COL = {
    "object_id": first_existing([
        "diaObjectId", "objectId", "object_id", "sourceId",
        "source_id", "alertId", "alert_id", "oid"
    ]),
    "time": first_existing([
        "midpointMjdTai", "mjd", "jd", "time", "timestamp"
    ]),
    "band": first_existing([
        "band", "filter", "filterName", "fid"
    ]),
    "flux": first_existing([
        "psfFlux", "differenceFlux", "diffFlux", "flux", "forcedDiffFlux"
    ]),
    "flux_err": first_existing([
        "psfFluxErr", "differenceFluxErr", "diffFluxErr",
        "flux_err", "fluxError", "forcedDiffFluxErr"
    ]),
    "mag": first_existing([
        "mag", "magnitude", "psfMag", "magpsf", "scienceMag"
    ]),
    "mag_err": first_existing([
        "magErr", "magnitude_error", "sigmapsf", "psfMagErr"
    ]),
    "ra": first_existing(["ra", "coord_ra", "ra_deg"]),
    "dec": first_existing(["dec", "decl", "coord_dec", "dec_deg"]),
}

print(json.dumps(COL, indent=2))

{
  "object_id": "oid",
  "time": "mjd",
  "band": null,
  "flux": "psfflux",
  "flux_err": "psffluxerr",
  "mag": null,
  "mag_err": null,
  "ra": null,
  "dec": null
}


### Manual column overrides


In [11]:
# Examples. Uncomment and edit only if needed.
# COL["object_id"] = "diaObjectId"
# COL["time"] = "midpointMjdTai"
# COL["band"] = "band"
# COL["flux"] = "psfFlux"
# COL["flux_err"] = "psfFluxErr"
# COL["mag"] = None
# COL["mag_err"] = None

if COL["object_id"] is None:
    raise KeyError("Set COL['object_id'] manually.")
if COL["flux"] is None and COL["mag"] is None:
    raise KeyError("Set COL['flux'] or COL['mag'] manually.")

print("Accepted mapping:")
print(json.dumps(COL, indent=2))

Accepted mapping:
{
  "object_id": "oid",
  "time": "mjd",
  "band": null,
  "flux": "psfflux",
  "flux_err": "psffluxerr",
  "mag": null,
  "mag_err": null,
  "ra": null,
  "dec": null
}


## 7. Fast object-level aggregation

This first-stage ranking uses inexpensive statistics. Robust quantiles can be calculated later for the leading candidates only.


In [12]:
def quote_identifier(name):
    if name is None:
        return None
    return '"' + str(name).replace('"', '""') + '"'


q = {key: quote_identifier(value) for key, value in COL.items()}
measurement_key = "flux" if COL["flux"] is not None else "mag"
measurement_sql = q[measurement_key]
object_id_sql = q["object_id"]

prepared_columns = [
    f"CAST({object_id_sql} AS VARCHAR) AS object_id",
    f"TRY_CAST({measurement_sql} AS DOUBLE) AS measurement",
]

if q["time"] is not None:
    prepared_columns.append(
        f"TRY_CAST({q['time']} AS DOUBLE) AS observation_time"
    )
if q["band"] is not None:
    prepared_columns.append(f"{q['band']} AS band")
if q["flux"] is not None and q["flux_err"] is not None:
    prepared_columns.extend([
        f"TRY_CAST({q['flux']} AS DOUBLE) AS flux",
        f"TRY_CAST({q['flux_err']} AS DOUBLE) AS flux_error",
    ])
if q["ra"] is not None:
    prepared_columns.append(f"TRY_CAST({q['ra']} AS DOUBLE) AS ra")
if q["dec"] is not None:
    prepared_columns.append(f"TRY_CAST({q['dec']} AS DOUBLE) AS dec")

aggregate_expressions = [
    "object_id",
    "count(*) AS n_observations",
    "avg(measurement) AS measurement_mean",
    "stddev_samp(measurement) AS measurement_std",
    "min(measurement) AS measurement_min",
    "max(measurement) AS measurement_max",
    "max(measurement) - min(measurement) AS measurement_range",
]

if q["time"] is not None:
    aggregate_expressions.extend([
        "min(observation_time) AS time_min",
        "max(observation_time) AS time_max",
        "max(observation_time) - min(observation_time) AS time_baseline",
    ])
if q["band"] is not None:
    aggregate_expressions.append("count(DISTINCT band) AS n_bands")
if q["flux"] is not None and q["flux_err"] is not None:
    aggregate_expressions.extend([
        "avg(flux / NULLIF(flux_error, 0)) AS snr_mean",
        "max(abs(flux / NULLIF(flux_error, 0))) AS snr_max_abs",
    ])
if q["ra"] is not None:
    aggregate_expressions.append("avg(ra) AS ra_mean")
if q["dec"] is not None:
    aggregate_expressions.append("avg(dec) AS dec_mean")

feature_query = f"""
WITH prepared AS (
    SELECT {', '.join(prepared_columns)}
    FROM rubin_alerts
    WHERE {object_id_sql} IS NOT NULL
)
SELECT {', '.join(aggregate_expressions)}
FROM prepared
WHERE measurement IS NOT NULL
GROUP BY object_id
HAVING count(*) >= {int(MIN_OBSERVATIONS)}
"""

print("Running aggregation on one Parquet file...")
start = perf_counter()
features = connection.execute(feature_query).df().set_index("object_id")
features = features.replace([np.inf, -np.inf], np.nan)
print(f"Completed in {perf_counter() - start:.1f} seconds")
print("Object-level feature matrix:", features.shape)
display(features.head())

Running aggregation on one Parquet file...
Completed in 0.0 seconds
Object-level feature matrix: (0, 11)


,n_observations,measurement_mean,measurement_std,measurement_min,measurement_max,measurement_range,time_min,time_max,time_baseline,snr_mean,snr_max_abs
object_id,,,,,,,,,,,


In [ ]:
placeholders = ", ".join(["?"] * len(top_ids))
robust_query = f"""
WITH selected AS (
    SELECT
        CAST({object_id_sql} AS VARCHAR) AS object_id,
        TRY_CAST({measurement_sql} AS DOUBLE) AS measurement
    FROM rubin_alerts
    WHERE CAST({object_id_sql} AS VARCHAR) IN ({placeholders})
)
SELECT
    object_id,
    approx_quantile(measurement, 0.50) AS robust_median,
    approx_quantile(measurement, 0.75)
        - approx_quantile(measurement, 0.25) AS robust_iqr,
    approx_quantile(measurement, 0.95)
        - approx_quantile(measurement, 0.05) AS robust_amplitude90
FROM selected
WHERE measurement IS NOT NULL
GROUP BY object_id
"""

robust_candidate_features = (
    connection.execute(robust_query, top_ids).df().set_index("object_id")
)
ranked = ranked.join(robust_candidate_features, how="left")
display(ranked.head(TOP_N))